In [ ]:
## 1. Basic Setup
# Updated with improved Silo caching - prevents rebuilding Silo every time!
# Now properly detects existing Silo installations using the MFC venv's Python version

import sys
import subprocess
import os
from pathlib import Path
import tempfile

def setup_mfc_environment(install_silo=True):
    """Initialize MFC and install notebook packages, optionally install Silo
    
    Enhanced with intelligent Silo caching that:
    - Uses the exact Python version from MFC's virtual environment
    - Checks for complete Silo installation (libraries + Python bindings)
    - Sets up persistent Python paths via .pth files
    - Prevents unnecessary rebuilds
    """
    print("=== MFC Environment Setup ===\n")
    
    # Check if we're in MFC root directory
    if not os.path.exists("mfc.sh"):
        print("❌ Error: mfc.sh not found")
        print("   Make sure you're running this notebook from the MFC root directory")
        return False, None
    
    print("🔧 Step 1: Initializing MFC virtual environment...")
    print("   Running: ./mfc.sh init")
    
    # Initialize MFC - creates virtual environment
    try:
        result = subprocess.run("./mfc.sh init", shell=True, capture_output=True, 
                              text=True, timeout=6000)
        if result.returncode != 0:
            print("❌ MFC initialization failed")
            print(f"   Error: {result.stderr}")
            return False, None
        print("✅ MFC initialized successfully\n")
    except subprocess.TimeoutExpired:
        print("❌ MFC initialization timed out")
        return False, None
    except Exception as e:
        print(f"❌ MFC initialization error: {e}")
        return False, None
    
    # Find MFC's virtual environment
    build_dir = Path("build")
    venv_path = None
    
    if build_dir.exists():
        for potential_venv in ["venv", ".venv", "env"]:
            candidate = build_dir / potential_venv
            if candidate.exists():
                venv_path = candidate
                break
    
    if not venv_path:
        print("❌ Could not find MFC's virtual environment")
        return False, None
    
    print(f"✅ Found MFC virtual environment: {venv_path}")
    
    # Get Python executable
    python_exe = venv_path / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    
    if not python_exe.exists():
        print(f"❌ Python executable not found: {python_exe}")
        return False, None
    
    # Install basic packages
    packages = ["matplotlib", "pandas", "h5py", "seaborn", "ipywidgets"]
    
    print("📦 Step 2: Installing notebook packages in MFC environment...")
    for package in packages:
        try:
            result = subprocess.run([str(python_exe), "-m", "pip", "install", package],
                                  capture_output=True, text=True, timeout=1200)
            if result.returncode == 0:
                print(f"   ✅ {package}")
            else:
                print(f"   ⚠️  {package} (failed)")
        except Exception as e:
            print(f"   ⚠️  {package} (error: {e})")
    
    # Optionally install Silo
    silo_success = False
    if install_silo:
        print("Step 3: Installing Silo library for MFC file analysis...")
        
        # Get Python version info from MFC's venv (crucial for proper caching)
        print("   Detecting Python version from MFC virtual environment...")
        result = subprocess.run([str(python_exe), "-c", 
                               "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
                               capture_output=True, text=True)
        python_version = result.stdout.strip() if result.returncode == 0 else "3.12"
        print(f"   Using Python {python_version} from MFC venv")
        
        # Check if Silo is already properly installed using the MFC venv's exact Python version
        print("   Checking if Silo is already installed...")
        silo_success = check_existing_silo_installation(
            python_exe, venv_path.absolute(), python_version
        )
        
        if not silo_success:
            print("   Silo not found or incomplete, proceeding with installation...")
            try:
                silo_success = install_silo_library(venv_path, python_exe)
            except KeyboardInterrupt:
                print("   ⚠️  Silo installation interrupted by user")
                print("   Continuing without Silo - basic analysis will still work")
                silo_success = False
    else:
        print("️  Step 3: Skipping Silo installation (can install later)")
        print("   Note: Advanced Silo file analysis will not be available")
    
    return True, str(python_exe)

def check_existing_silo_installation(python_exe, venv_abs, python_version):
    """Check if Silo is already properly installed using the MFC venv's exact Python version"""
    
    # Expected Silo installation directory (where install_silo_library puts it)
    silo_install_dir = venv_abs / "silo_install"
    
    # Check if installation directory exists
    if not silo_install_dir.exists():
        print(f"   Silo installation directory not found: {silo_install_dir}")
        return False
    
    # Check for key Silo files to verify installation completeness
    # Focus on library files that are essential for the Python module
    # Note: Silo may create different library files depending on build config
    expected_files = [
        # Check for any of the main Silo library files that might exist
        silo_install_dir / "lib" / "libsilo.a",      # Standard Silo
        silo_install_dir / "lib" / "libsiloh5.a",    # Silo with HDF5 support
        silo_install_dir / "lib" / "Silo.a",         # Python module library
    ]
    
    # Check for Python module files
    python_module_files = [
        silo_install_dir / "lib" / "Silo.so",
        silo_install_dir / "lib" / "Silo.dylib",
        silo_install_dir / "lib" / "_Silo.so",
    ]
    
    # At least one library file should exist
    existing_libs = [f for f in expected_files if f.exists()]
    if not existing_libs:
        print(f"   No Silo library files found. Checked: {[str(f) for f in expected_files]}")
        return False
    else:
        print(f"   Found Silo library files: {[str(f) for f in existing_libs]}")
        
    # At least one Python module file should exist
    has_python_module = any(f.exists() for f in python_module_files)
    if not has_python_module:
        print(f"   Missing Silo Python module files: {[str(f) for f in python_module_files]}")
        return False
    
    # Find Silo Python module paths using the exact Python version from MFC venv
    silo_python_paths = find_silo_python_paths(silo_install_dir, python_version)
    
    if not silo_python_paths:
        print(f"   Silo Python module not found for Python {python_version}")
        return False
    
    # Set up the .pth file for persistent access
    setup_silo_python_path(silo_python_paths, venv_abs, python_version)
    
    # Test Silo import using the MFC venv's Python with proper PYTHONPATH
    env = os.environ.copy()
    pythonpath = ":".join(str(p) for p in silo_python_paths)
    existing_pythonpath = env.get('PYTHONPATH', '')
    env['PYTHONPATH'] = f"{pythonpath}:{existing_pythonpath}" if existing_pythonpath else pythonpath
    
    result = subprocess.run([str(python_exe), "-c", "import Silo; print('Silo import successful')"],
                          capture_output=True, text=True, env=env, timeout=10)
    
    if result.returncode == 0:
        print(f"   ✅ Silo already installed and working: {result.stdout.strip()}")
        return True
    else:
        print(f"   Silo import test failed: {result.stderr}")
        return False

def find_silo_python_paths(silo_install_dir, python_version):
    """Find Silo Python module locations for the specific Python version"""
    # These are the paths where Silo's configure/make install might put the Python module
    # Note: Silo often installs directly in lib/ rather than python-version-specific dirs
    possible_paths = [
        silo_install_dir / "lib",  # Check lib/ first - this is where Silo.so typically goes
        silo_install_dir / "lib" / f"python{python_version}" / "site-packages",
        silo_install_dir / "lib" / "python" / "site-packages", 
        silo_install_dir / f"lib/python{python_version}/site-packages",
        silo_install_dir / "lib64" / f"python{python_version}" / "site-packages",
        # Also check for the 'm' suffix variant (e.g., python3.12m)
        silo_install_dir / "lib" / f"python{python_version}m" / "site-packages",
    ]
    
    existing_paths = []
    for path in possible_paths:
        if path.exists():
            # Verify Silo files are actually there (look for .so, .a, or .py files)
            silo_files = (list(path.glob("*Silo*")) + 
                         list(path.glob("*silo*")) + 
                         list(path.glob("Silo.so")) + 
                         list(path.glob("Silo.a")))
            if silo_files:
                print(f"   Found Silo Python module in: {path}")
                existing_paths.append(path)
    
    return existing_paths

def setup_silo_python_path(silo_python_paths, venv_abs, python_version):
    """Set up persistent Python path for Silo in the MFC venv"""
    if not silo_python_paths:
        return False
    
    # Create .pth file in the MFC venv's site-packages for persistent access
    venv_site_packages = venv_abs / "lib" / f"python{python_version}" / "site-packages"
    
    # Also check for the 'm' suffix variant
    if not venv_site_packages.exists():
        venv_site_packages = venv_abs / "lib" / f"python{python_version}m" / "site-packages"
    
    if venv_site_packages.exists():
        pth_file = venv_site_packages / "silo.pth"
        
        with open(pth_file, 'w') as f:
            for path in silo_python_paths:
                f.write(f"{path}\n")
        
        print(f"   Updated Silo path file: {pth_file}")
        return True
    else:
        print(f"   Warning: Could not find MFC venv site-packages for Python {python_version}")
        return False

def install_silo_library(venv_path, python_exe):
    """Install Silo library with Python bindings for MFC file analysis"""
    
    # Check for required build tools first
    required_tools = ["git", "make", "gcc", "g++"]
    missing_tools = []
    
    for tool in required_tools:
        result = subprocess.run(["which", tool], capture_output=True)
        if result.returncode != 0:
            missing_tools.append(tool)
    
    if missing_tools:
        print(f"   ❌ Missing build tools: {', '.join(missing_tools)}")
        print("   Install with: brew install git make gcc (on macOS)")
        return False
    
    try:
        # Get Python version info from MFC's venv
        result = subprocess.run([str(python_exe), "-c", 
                               "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
                               capture_output=True, text=True)
        python_version = result.stdout.strip() if result.returncode == 0 else "3.12"
        
        # Find Python include directory - USE ABSOLUTE PATHS
        venv_abs = venv_path.absolute()
        python_exe_abs = python_exe.absolute()
        python_include = venv_abs / f"include/python{python_version}"
        
        if not python_include.exists():
            for alt in [venv_abs / "include", venv_abs / f"include/python{python_version}m"]:
                if alt.exists():
                    python_include = alt
                    break
        
        python_include_abs = python_include.absolute()
        print(f"   Using Python {python_version}, include: {python_include_abs}")
        
        # Create temporary directory for building Silo
        with tempfile.TemporaryDirectory() as temp_dir:
            temp_path = Path(temp_dir)
            silo_dir = temp_path / "Silo"
            
            # Clone Silo repository
            print("   Cloning Silo repository...")
            result = subprocess.run(
                ["git", "clone", "--depth=1", "https://github.com/LLNL/Silo.git", str(silo_dir)],
                capture_output=True, text=True, timeout=3000
            )
            
            if result.returncode != 0:
                print(f"   ❌ Failed to clone Silo: {result.stderr}")
                return False
            
            print("   ✅ Silo repository cloned")
            
            # Configure Silo with ABSOLUTE PATHS
            print("   Configuring Silo...")
            silo_prefix_abs = (venv_abs / "silo_install").absolute()
            
            # Check for HDF5 installation (needed for MFC's Silo files)
            hdf5_path = None
            for hdf5_check in ["/opt/homebrew", "/usr/local", "/usr"]:
                if Path(f"{hdf5_check}/include/hdf5.h").exists():
                    hdf5_path = hdf5_check
                    break
            
            configure_cmd = [
                "./configure",
                f"--prefix={silo_prefix_abs}",
                "--enable-pythonmodule",
                f"--with-python={python_exe_abs}",
                f"--includedir={python_include_abs}",
                "--with-hdf5"
            ]
            
            # Add HDF5 path if found
            if hdf5_path:
                configure_cmd.extend([f"--with-hdf5={hdf5_path}", f"CPPFLAGS=-I{hdf5_path}/include", f"LDFLAGS=-L{hdf5_path}/lib"])
                print(f"   Using HDF5 from: {hdf5_path}")
            else:
                print("   Warning: HDF5 not found - trying system default")
                # Try to find HDF5 via pkg-config or use system default
                try:
                    result = subprocess.run(["brew", "--prefix", "hdf5"], capture_output=True, text=True)
                    if result.returncode == 0:
                        brew_hdf5 = result.stdout.strip()
                        configure_cmd.extend([f"--with-hdf5={brew_hdf5}"])
                        print(f"   Found HDF5 via brew: {brew_hdf5}")
                except:
                    pass
            
            print(f"   Configure command: {' '.join(configure_cmd)}")
            
            result = subprocess.run(
                configure_cmd, cwd=silo_dir, capture_output=True, text=True, timeout=6000
            )
            
            if result.returncode != 0:
                print(f"   ❌ Silo configure failed")
                print(f"   Configure output: {result.stderr[-500:]}")
                return False
            
            print("   ✅ Silo configured successfully")
            
            # Build and install Silo
            print("   Building Silo (this takes several minutes)...")
            result = subprocess.run(
                ["make", "-j10"], cwd=silo_dir, capture_output=True, text=True, timeout=18000
            )
            
            if result.returncode != 0:
                print(f"   ❌ Silo build failed")
                print(f"   Build error: {result.stderr[-500:]}")
                return False
            
            print("   ✅ Silo built successfully")
            
            print("   Installing Silo...")
            result = subprocess.run(
                ["make", "install"], cwd=silo_dir, capture_output=True, text=True, timeout=30000
            )
            
            if result.returncode != 0:
                print(f"   ❌ Silo install failed")
                print(f"   Install error: {result.stderr[-500:]}")
                return False
            
            print("   ✅ Silo installed successfully")
            
            # Find and add Silo to Python path - check multiple possible locations
            silo_installed = False
            possible_python_paths = [
                silo_prefix_abs / "lib" / f"python{python_version}" / "site-packages",
                silo_prefix_abs / "lib" / "python" / "site-packages", 
                silo_prefix_abs / f"lib/python{python_version}/site-packages",
                silo_prefix_abs / "lib",
                silo_prefix_abs / "lib64" / f"python{python_version}" / "site-packages"
            ]
            
            for silo_python_path in possible_python_paths:
                if silo_python_path.exists():
                    # Check if Silo files are actually there
                    silo_files = list(silo_python_path.glob("*Silo*")) + list(silo_python_path.glob("*silo*"))
                    if silo_files:
                        print(f"   Found Silo in: {silo_python_path}")
                        
                        # Add to PYTHONPATH via .pth file
                        pth_file = venv_abs / "lib" / f"python{python_version}" / "site-packages" / "silo.pth"
                        pth_file.parent.mkdir(parents=True, exist_ok=True)
                        with open(pth_file, 'w') as f:
                            f.write(str(silo_python_path))
                        
                        # Also try setting PYTHONPATH environment variable for the test
                        env = os.environ.copy()
                        current_pythonpath = env.get('PYTHONPATH', '')
                        if current_pythonpath:
                            env['PYTHONPATH'] = f"{silo_python_path}:{current_pythonpath}"
                        else:
                            env['PYTHONPATH'] = str(silo_python_path)
                        
                        # Test Silo import with explicit PYTHONPATH
                        result = subprocess.run([str(python_exe_abs), "-c", "import Silo; print('Silo import successful')"],
                                              capture_output=True, text=True, env=env)
                        
                        if result.returncode == 0:
                            print(f"   ✅ Silo Python module working: {result.stdout.strip()}")
                            silo_installed = True
                            break
                        else:
                            print(f"   Testing path {silo_python_path}: {result.stderr.strip()}")
            
            if not silo_installed:
                print(f"   ⚠️  Silo installed but Python module not found in expected locations")
                print(f"   Checked: {[str(p) for p in possible_python_paths]}")
                return False
            
            return True
                
    except Exception as e:
        print(f"   ❌ Silo installation error: {e}")
        return False

# Setup options - change install_silo to True if you want automatic Silo installation
INSTALL_SILO = True  # Set to True for full Silo support (takes 10+ minutes)

print("🚀 Starting MFC environment setup...")
print(f"   Silo installation: {'Enabled' if INSTALL_SILO else 'Disabled (can enable later)'}")
print("   💡 To enable Silo: set INSTALL_SILO = True above and re-run this cell")
print("")

success, mfc_python = setup_mfc_environment(install_silo=INSTALL_SILO)

if success:
    print(f"  🎉 Setup Complete!")
    print(f"   MFC Python: {mfc_python}")
    
    # Import packages and set up environment
    import json, glob, time, warnings
    import numpy as np
    
    # Set global flags for available packages
    MFC_ROOT = os.getcwd()
    PACKAGES = {}
    
    for pkg_name, import_name in [("matplotlib", "matplotlib.pyplot"), ("pandas", "pandas"), 
                                  ("h5py", "h5py"), ("seaborn", "seaborn"), ("ipywidgets", "ipywidgets")]:
        try:
            __import__(import_name)
            PACKAGES[pkg_name] = True
            print(f"   ✅ {pkg_name} available")
        except ImportError:
            PACKAGES[pkg_name] = False
            print(f"   ❌ {pkg_name} not available")
    
    # Check for Silo specifically - test in MFC's Python environment
    try:
        # Test Silo import in MFC's venv (where it was installed)
        result = subprocess.run([mfc_python, "-c", "import Silo; print('Silo available')"],
                              capture_output=True, text=True, timeout=1000)
        if result.returncode == 0:
            PACKAGES["silo"] = True
            print(f"   ✅ Silo available")
        else:
            PACKAGES["silo"] = False
            print(f"   ❌ Silo import failed in MFC environment: {result.stderr}")
    except Exception as e:
        PACKAGES["silo"] = False
        print(f"   ❌ Silo test failed: {e}")
        print(f"   ℹ️  Set INSTALL_SILO=True to enable Silo installation")
    
    # Set up plotting if available
    if PACKAGES["matplotlib"]:
        import matplotlib.pyplot as plt
        if PACKAGES["seaborn"]:
            import seaborn as sns
            sns.set_palette("husl")
            plt.style.use('seaborn-v0_8')
        else:
            plt.style.use('default')
    
    warnings.filterwarnings('ignore')
    
else:
    print("  ❌ Setup failed. Check errors above and ensure:")
    print("   1. You're in the MFC root directory")
    print("   2. You have internet access")
    print("   3. For Silo: build tools (git, make, gcc)")

In [ ]:
## 2. MFC Build System

def build_mfc(targets=['pre_process', 'simulation', 'post_process'], mpi=True, jobs=None):
    """Build MFC with specified options - includes post_process for Silo file generation"""
    # Check if we have the success variable from setup
    try:
        if not success:
            print("❌ Cannot build - environment setup failed")
            return False
    except NameError:
        print("❌ Cannot build - run the setup cell first")
        return False
    
    print(f"=== Building MFC ===")
    print(f"Targets: {', '.join(targets)}")
    print("ℹ️  Including post_process for Silo visualization data generation")
    
    # Determine number of jobs
    if jobs is None:
        try:
            jobs = int(subprocess.run("nproc", shell=True, capture_output=True, text=True).stdout.strip())
        except:
            jobs = 4
    
    # Build command
    cmd = f"./mfc.sh build -j {jobs} -t {' '.join(targets)}"
    if mpi:
        cmd += " --mpi"
    
    print(f"Running: {cmd}")
    
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
        if result.returncode == 0:
            print("✅ MFC build successful!")
            print("✅ All components built: pre_process, simulation, post_process")
            return True
        else:
            print("❌ MFC build failed")
            print(f"Error: {result.stderr[-500:]}")
            return False
    except subprocess.TimeoutExpired:
        print("❌ Build timed out after 30 minutes")
        return False

# Build MFC if environment setup was successful
try:
    if success:
        print("🔨 Starting MFC build...")
        build_success = build_mfc()
    else:
        build_success = False
        print("Skipping build due to environment setup failure")
except NameError:
    print("❌ Setup variables not found - run the setup cell first")
    build_success = False

In [ ]:
## 3. Example Runner and Analysis
## Run examples and analyze results with or without Silo. If you have `INSTALL_SILO=True`, you'll get full Silo-based analysis. Otherwise, basic file analysis is provided.

def list_examples():
    """List all available MFC examples"""
    examples_dir = Path("examples")
    examples = []
    
    if examples_dir.exists():
        for item in examples_dir.iterdir():
            case_file = item / "case.py"
            if item.is_dir() and case_file.exists():
                examples.append({
                    'name': item.name,
                    'path': str(item),
                    'case_file': str(case_file),
                    'dimension': item.name[:2] if item.name[0].isdigit() else 'Unknown'
                })
    
    return sorted(examples, key=lambda x: (x['dimension'], x['name']))

def run_example(example_name, stages=['pre_process', 'simulation', 'post_process'], num_procs=2, timeout=600):
    """Run a specific MFC example - includes post_process by default for Silo file generation"""
    if not build_success:
        print("❌ Cannot run examples - MFC build failed")
        return None
    
    # Find available examples
    examples_list = list_examples()
    
    # Find the example
    example_path = None
    for ex in examples_list:
        if ex['name'] == example_name:
            example_path = ex['case_file']
            break
    
    if not example_path:
        print(f"❌ Example '{example_name}' not found!")
        print("Available examples:")
        for ex in examples_list[:10]:
            print(f"  • {ex['name']}")
        return None
    
    print(f"=== Running Example: {example_name} ===")
    print(f"Stages: {', '.join(stages)}")
    print(f"Processors: {num_procs}")
    
    if 'post_process' in stages:
        print("✅ Post-processing enabled - will generate Silo files for visualization")
    else:
        print("⚠️  Post-processing disabled - no Silo visualization files will be generated")
    
    cmd = f"./mfc.sh run {example_path} -n {num_procs} -t {' '.join(stages)} --no-build"
    print(f"Running: {cmd}")
    
    start_time = time.time()
    
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        runtime = time.time() - start_time
        
        if result.returncode == 0:
            print(f"✅ Example completed successfully in {runtime:.1f}s")
            
            # Check for output files
            example_dir = Path(example_path).parent
            output_dirs = ['silo_hdf5', 'silo', 'hdf5']
            
            silo_files_found = False
            for output_dir in output_dirs:
                full_path = example_dir / output_dir
                if full_path.exists():
                    files = []
                    silo_count = 0
                    for item in full_path.iterdir():
                        if item.is_file():
                            files.append(item)
                            if item.suffix in ['.silo', '.h5', '.hdf5']:
                                silo_count += 1
                        elif item.is_dir():
                            # Count files in subdirectories too (MFC stores files in subdirs like root/, p0/, p1/)
                            subdir_files = list(item.iterdir())
                            files.extend(subdir_files)
                            silo_count += len([f for f in subdir_files if f.suffix in ['.silo', '.h5', '.hdf5']])
                    print(f"   Output: {output_dir}/ ({len(files)} files, {silo_count} Silo/HDF5)")
                    if silo_count > 0:
                        silo_files_found = True
            
            if silo_files_found:
                print("✅ Silo/HDF5 visualization files generated successfully")
            elif 'post_process' in stages:
                print("⚠️  Post-processing ran but no Silo files found")
            else:
                print("ℹ️  No Silo files (post-processing not included)")
            
            return {
                'success': True,
                'runtime': runtime,
                'example_dir': str(example_dir),
                'stdout': result.stdout,
                'stderr': result.stderr
            }
        else:
            print(f"❌ Example failed after {runtime:.1f}s")
            print(f"Error: {result.stderr[-300:]}")
            return {'success': False, 'runtime': runtime, 'stderr': result.stderr}
            
    except subprocess.TimeoutExpired:
        print(f"❌ Example timed out after {timeout}s")
        return {'success': False, 'runtime': timeout, 'stderr': 'Timeout'}

def analyze_time_series(run_result, selected_timesteps=None):
    """Analyze MFC results across multiple time steps"""
    if not run_result or not run_result['success']:
        print("No successful run to analyze")
        return
    
    print(f"=== Time Series Analysis ===")
    print(f"Runtime: {run_result['runtime']:.2f} seconds")
    
    example_dir = Path(run_result['example_dir'])
    
    # Find all collection files and extract time steps
    collection_files = []
    for output_dir in ['silo_hdf5', 'silo', 'hdf5']:
        dir_path = example_dir / output_dir
        if dir_path.exists():
            root_dir = dir_path / 'root'
            if root_dir.exists():
                files = list(root_dir.glob('collection_*.silo'))
                collection_files.extend(files)
    
    if not collection_files:
        print("❌ No collection files found for time series analysis")
        return
    
    # Extract time steps from filenames
    time_steps = []
    file_map = {}
    for file in collection_files:
        try:
            # Extract number from collection_XXX.silo
            time_str = file.stem.split('_')[1]
            time_step = int(time_str)
            time_steps.append(time_step)
            file_map[time_step] = file
        except:
            continue
    
    time_steps.sort()
    print(f"📊 Found {len(time_steps)} time steps: {time_steps[:10]}{'...' if len(time_steps) > 10 else ''}")
    
    # Select time steps to analyze
    if selected_timesteps is None:
        # Analyze all time steps unless there are more than 100
        if len(time_steps) <= 100:
            selected_timesteps = time_steps
            print(f"🎯 Will analyze all {len(time_steps)} time steps")
        else:
            # Cap at 100 evenly spaced time steps
            step = max(1, len(time_steps) // 100)
            selected_timesteps = time_steps[::step][:100]
            print(f"🎯 Too many time steps ({len(time_steps)}), capping at 100 evenly spaced")
    
    # Analyze each selected time step
    time_series_data = {}
    for ts in selected_timesteps:
        if ts in file_map:
            print(f"\n⏱️  Time step {ts}:")
            try:
                data = analyze_single_timestep(file_map[ts])
                if data:
                    time_series_data[ts] = data
                    dimensionality = data.get('dimensionality', '1D')
                    print(f"   ✅ Extracted {len(data['variables'])} {dimensionality} variables")
            except Exception as e:
                print(f"   ❌ Failed to analyze time step {ts}: {e}")
    
    # Create time series plots
    if time_series_data and PACKAGES.get('matplotlib'):
        plot_time_series(time_series_data)
    
    return time_series_data

def analyze_single_timestep(file_path):
    """Analyze a single time step file and return variable data with 2D support"""
    try:
        import h5py
        import numpy as np
        
        with h5py.File(file_path, 'r') as f:
            # Enhanced variable mapping extraction for 2D support
            variable_mappings = {}
            variable_refs = {}  # Store the full reference structure
            
            for key in f.keys():
                try:
                    group = f[key]
                    if hasattr(group, 'attrs') and 'silo' in group.attrs:
                        silo_attr = group.attrs['silo']
                        if hasattr(silo_attr, 'item') and len(silo_attr.item()) >= 3:
                            silo_tuple = silo_attr.item()
                            # For 2D data, we might have multiple references
                            if len(silo_tuple) >= 7:  # 2D case with nvars=2
                                nvars = silo_tuple[0]
                                if nvars == 2:  # 2D case
                                    ref1 = silo_tuple[4].decode('utf-8') if isinstance(silo_tuple[4], bytes) else str(silo_tuple[4])
                                    ref2 = silo_tuple[5].decode('utf-8') if isinstance(silo_tuple[5], bytes) else str(silo_tuple[5])
                                    ref3 = silo_tuple[6].decode('utf-8') if isinstance(silo_tuple[6], bytes) else str(silo_tuple[6])
                                    variable_refs[key] = {
                                        'type': '2D',
                                        'nvars': nvars,
                                        'grid_refs': [ref1, ref2],
                                        'data_ref': ref3
                                    }
                                    variable_mappings[ref3] = key
                                else:  # 1D case or other
                                    data_ref = silo_tuple[2].decode('utf-8') if isinstance(silo_tuple[2], bytes) else str(silo_tuple[2])
                                    variable_refs[key] = {
                                        'type': '1D',
                                        'data_ref': data_ref
                                    }
                                    variable_mappings[data_ref] = key
                            else:  # Standard 1D case
                                data_ref = silo_tuple[2].decode('utf-8') if isinstance(silo_tuple[2], bytes) else str(silo_tuple[2])
                                variable_refs[key] = {
                                    'type': '1D',
                                    'data_ref': data_ref
                                }
                                variable_mappings[data_ref] = key
                        elif isinstance(silo_attr, (list, tuple)) and len(silo_attr) >= 3:
                            data_ref = silo_attr[2].decode('utf-8') if isinstance(silo_attr[2], bytes) else str(silo_attr[2])
                            variable_refs[key] = {
                                'type': '1D',
                                'data_ref': data_ref
                            }
                            variable_mappings[data_ref] = key
                except:
                    continue
            
            # Collect all datasets
            found_vars = []
            var_info = {}
            
            def find_datasets(name, obj):
                if isinstance(obj, h5py.Dataset):
                    found_vars.append(name)
                    shape = obj.shape
                    dtype = obj.dtype
                    
                    if obj.size > 0:
                        try:
                            data_sample = obj[...]
                            var_info[name] = {
                                'name': name,
                                'shape': shape,
                                'dtype': str(dtype),
                                'data': data_sample
                            }
                            if hasattr(data_sample, 'min'):
                                var_info[name]['min'] = float(data_sample.min())
                                var_info[name]['max'] = float(data_sample.max())
                        except:
                            var_info[name] = {
                                'name': name,
                                'shape': shape,
                                'dtype': str(dtype),
                                'data': None,
                                'min': None,
                                'max': None
                            }
            
            f.visititems(find_datasets)
            
            # Process variables based on dimensionality
            result = {
                'variables': {},
                'grid': None,
                'grid_2d': None,
                'dimensionality': '1D',
                'file': file_path.name
            }
            
            # Check if we have any 2D variables
            has_2d = any(ref_info['type'] == '2D' for ref_info in variable_refs.values())
            
            if has_2d:
                result['dimensionality'] = '2D'
                num_2d_vars = sum(1 for ref_info in variable_refs.values() if ref_info['type'] == '2D')
                print(f"   🎯 Detected 2D data structure with {num_2d_vars} variables")
                
                # Helper function to find datasets
                def find_dataset(ref):
                    if ref in var_info:
                        return ref
                    elif ref.lstrip('/') in var_info:
                        return ref.lstrip('/')
                    elif '/' + ref.lstrip('/') in var_info:
                        return '/' + ref.lstrip('/')
                    return None
                
                # First, look for rectilinear_grid to use as the main coordinate system
                global_grid_x = None
                global_grid_y = None
                
                # Check if we have a rectilinear_grid variable
                if 'rectilinear_grid' in variable_refs:
                    ref_info = variable_refs['rectilinear_grid']
                    if ref_info['type'] == '2D':
                        try:
                            data_ref = ref_info['data_ref']
                            data_path = find_dataset(data_ref)
                            
                            if data_path:
                                grid_data = var_info[data_path]['data']
                                if grid_data is not None and grid_data.ndim == 2:
                                    print(f"   🔍 Analyzing rectilinear_grid shape: {grid_data.shape}")
                                    print(f"   📊 Grid data preview:\n{grid_data}")
                                    print(f"   📈 Grid data range: [{grid_data.min():.3f}, {grid_data.max():.3f}]")
                                    
                                    # Helper function to extract unique sorted coordinates
                                    def extract_unique_coords(data):
                                        return np.unique(data)
                                    
                                    # Try multiple interpretation strategies
                                    strategies = []
                                    
                                    # Strategy 1: Rows are x,y coordinates
                                    if grid_data.shape[0] == 2:
                                        x_coords = extract_unique_coords(grid_data[0, :])
                                        y_coords = extract_unique_coords(grid_data[1, :])
                                        strategies.append(("rows", x_coords, y_coords))
                                    
                                    # Strategy 2: Columns are x,y coordinates  
                                    if grid_data.shape[1] == 2:
                                        x_coords = extract_unique_coords(grid_data[:, 0])
                                        y_coords = extract_unique_coords(grid_data[:, 1])
                                        strategies.append(("columns", x_coords, y_coords))
                                    
                                    # Strategy 3: Grid might be flattened cell centers - try to reconstruct
                                    if grid_data.size >= 4:
                                        flat_data = grid_data.flatten()
                                        x_coords = extract_unique_coords(flat_data)
                                        # Assume square grid for now
                                        n_points = int(np.sqrt(len(x_coords)))
                                        if n_points * n_points == len(x_coords):
                                            strategies.append(("square_grid", x_coords[:n_points], x_coords[:n_points]))
                                    
                                    # Strategy 4: If data looks like cell boundaries, convert to [0,1]
                                    # Common case: data might be in range [-0.5, 0.5] and needs shifting
                                    for strategy_name, x_cand, y_cand in strategies:
                                        print(f"   🧪 Testing {strategy_name} - X: [{x_cand.min():.3f}, {x_cand.max():.3f}] ({len(x_cand)} points), Y: [{y_cand.min():.3f}, {y_cand.max():.3f}] ({len(y_cand)} points)")
                                        
                                        # Try direct coordinates first
                                        if 0 <= x_cand.min() <= x_cand.max() <= 1 and 0 <= y_cand.min() <= y_cand.max() <= 1:
                                            global_grid_x = x_cand
                                            global_grid_y = y_cand
                                            print(f"   ✅ {strategy_name} direct: coordinates already in [0,1] domain")
                                            break
                                        
                                        # Try shifting coordinates to [0,1] range
                                        x_shifted = x_cand - x_cand.min()
                                        y_shifted = y_cand - y_cand.min()
                                        x_norm = x_shifted / x_shifted.max() if x_shifted.max() > 0 else x_shifted
                                        y_norm = y_shifted / y_shifted.max() if y_shifted.max() > 0 else y_shifted
                                        
                                        if 0 <= x_norm.min() <= x_norm.max() <= 1 and 0 <= y_norm.min() <= y_norm.max() <= 1:
                                            global_grid_x = x_norm
                                            global_grid_y = y_norm
                                            print(f"   ✅ {strategy_name} normalized: shifted to [0,1] domain")
                                            print(f"   🔧 Applied normalization: X: {x_cand.min():.3f}→0, {x_cand.max():.3f}→1")
                                            break
                                        
                                        # Try assuming it's centered around origin and needs shifting
                                        if abs(x_cand.mean()) < 0.1 and abs(y_cand.mean()) < 0.1:  # Close to origin
                                            x_centered = x_cand + 0.5  # Shift [-0.5,0.5] to [0,1]
                                            y_centered = y_cand + 0.5
                                            if 0 <= x_centered.min() <= x_centered.max() <= 1 and 0 <= y_centered.min() <= y_centered.max() <= 1:
                                                global_grid_x = x_centered
                                                global_grid_y = y_centered
                                                print(f"   ✅ {strategy_name} centered: shifted from origin to [0,1] domain")
                                                break
                                    
                                    if global_grid_x is not None:
                                        print(f"   📐 Using rectilinear_grid as coordinate system: x({len(global_grid_x)}), y({len(global_grid_y)})")
                                        print(f"   🎯 Final domain: X ∈ [{global_grid_x.min():.3f}, {global_grid_x.max():.3f}], Y ∈ [{global_grid_y.min():.3f}, {global_grid_y.max():.3f}]")
                                    else:
                                        print(f"   ⚠️  Could not determine correct grid interpretation from any strategy")
                                        print(f"   🔍 Consider manually inspecting the grid structure")
                                else:
                                    print(f"   ⚠️  rectilinear_grid has unexpected dimensions or is None")
                        except Exception as e:
                            print(f"   ⚠️  Failed to extract rectilinear_grid: {e}")
                
                # Process 2D variables (excluding rectilinear_grid)
                for var_name, ref_info in variable_refs.items():
                    if ref_info['type'] == '2D' and var_name != 'rectilinear_grid':  # Skip rectilinear_grid
                        try:
                            # Get data reference
                            data_ref = ref_info['data_ref']
                            data_path = find_dataset(data_ref)
                            
                            if data_path:
                                var_data = var_info[data_path]['data']
                                
                                # ALWAYS use global grid if available (the corrected rectilinear_grid coordinates)
                                if global_grid_x is not None and global_grid_y is not None:
                                    grid_x = global_grid_x
                                    grid_y = global_grid_y
                                    print(f"   🎯 Using corrected global grid for {var_name}")
                                else:
                                    # Fallback to individual grid references
                                    grid_x_ref = ref_info['grid_refs'][0]
                                    grid_y_ref = ref_info['grid_refs'][1]
                                    grid_x_path = find_dataset(grid_x_ref)
                                    grid_y_path = find_dataset(grid_y_ref)
                                    
                                    if grid_x_path and grid_y_path:
                                        grid_x = var_info[grid_x_path]['data']
                                        grid_y = var_info[grid_y_path]['data']
                                        print(f"   ⚠️  Using fallback individual grid for {var_name} (may be incorrect)")
                                    else:
                                        print(f"   ⚠️  Could not find grid references for {var_name}")
                                        continue
                                
                                # Store 2D variable with the corrected coordinates
                                result['variables'][var_name] = {
                                    'data': var_data,
                                    'grid_x': grid_x,  # This should now be the corrected coordinates
                                    'grid_y': grid_y,  # This should now be the corrected coordinates
                                    'shape': var_data.shape if var_data is not None else None
                                }
                                
                                # Set up global 2D grid for plotting (using the rectilinear_grid if available)
                                if result['grid_2d'] is None:
                                    result['grid_2d'] = {
                                        'x': grid_x,
                                        'y': grid_y
                                    }
                                
                                print(f"   ✅ Extracted 2D variable: {var_name} with shape {var_data.shape if var_data is not None else 'None'}")
                            else:
                                print(f"   ⚠️  Could not find data for {var_name}")
                        except Exception as e:
                            print(f"   Failed to extract 2D variable {var_name}: {e}")
            else:
                # Process 1D variables (original logic)
                result['dimensionality'] = '1D'
                silo_datasets = [var for var in var_info.values() if '.silo/#' in var['name']]
                
                # Find datasets that have proper variable names
                mapped_variables = {}
                grid_data = None
                
                for dataset in silo_datasets:
                    dataset_name = dataset['name']
                    var_name = variable_mappings.get(dataset_name, variable_mappings.get('/' + dataset_name, dataset_name))
                    
                    if var_name != dataset_name:
                        # This dataset has a proper MFC variable name
                        mapped_variables[var_name] = dataset['data']
                    elif grid_data is None and dataset['data'] is not None:
                        # Use first unmapped dataset as grid
                        grid_data = dataset['data']
                
                result['variables'] = mapped_variables
                result['grid'] = grid_data
            
            return result
            
    except Exception as e:
        print(f"   Error reading {file_path.name}: {e}")
        return None

def plot_time_series(time_series_data):
    """Create time series plots showing variable evolution (handles both 1D and 2D data)"""
    import matplotlib.pyplot as plt
    import numpy as np
    
    if not time_series_data:
        print("No time series data to plot")
        return
    
    # Get all unique variables across time steps
    all_variables = set()
    for data in time_series_data.values():
        all_variables.update(data['variables'].keys())
    
    all_variables = sorted(list(all_variables))
    time_steps = sorted(time_series_data.keys())
    
    # Check if this is 2D data
    sample_data = next(iter(time_series_data.values()))
    is_2d = sample_data.get('dimensionality') == '2D'
    
    print(f"🎨 Creating {'2D' if is_2d else '1D'} time series plots for variables: {all_variables}")
    
    if is_2d:
        plot_2d_time_series(time_series_data, all_variables, time_steps)
    else:
        plot_1d_time_series(time_series_data, all_variables, time_steps)

def plot_1d_time_series(time_series_data, all_variables, time_steps):
    """Plot 1D time series (original functionality)"""
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Create subplots for each variable
    n_vars = len(all_variables)
    if n_vars == 0:
        print("No variables found for plotting")
        return
    
    fig, axes = plt.subplots(n_vars, 1, figsize=(14, 4*n_vars))
    if n_vars == 1:
        axes = [axes]
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(time_steps)))
    
    for i, var_name in enumerate(all_variables):
        ax = axes[i]
        
        for j, ts in enumerate(time_steps):
            if ts in time_series_data and var_name in time_series_data[ts]['variables']:
                data = time_series_data[ts]
                y_data = data['variables'][var_name]
                
                # Use grid data if available, otherwise use indices
                if data['grid'] is not None and len(data['grid']) == len(y_data):
                    x_data = data['grid']
                    ax.set_xlabel('Grid Position')
                else:
                    x_data = range(len(y_data))
                    ax.set_xlabel('Grid Index')
                
                ax.plot(x_data, y_data, color=colors[j], linewidth=2, 
                       label=f't={ts}', alpha=0.8)
        
        ax.set_ylabel(var_name)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print(f"✅ Time series plots created for {n_vars} variables")

def plot_2d_time_series(time_series_data, all_variables, time_steps):
    """Plot 2D time series using heatmaps and contour plots"""
    import matplotlib.pyplot as plt
    import numpy as np
    
    n_vars = len(all_variables)
    if n_vars == 0:
        print("No variables found for plotting")
        return
    
    # Show only a subset of time steps if there are too many
    max_time_steps = 6
    if len(time_steps) > max_time_steps:
        step = max(1, len(time_steps) // max_time_steps)
        selected_time_steps = time_steps[::step][:max_time_steps]
        print(f"   Showing {len(selected_time_steps)} time steps out of {len(time_steps)}")
    else:
        selected_time_steps = time_steps
    
    for var_name in all_variables:
        print(f"   📊 Creating 2D plots for {var_name}")
        
        # Create subplots for each time step
        n_times = len(selected_time_steps)
        cols = min(3, n_times)
        rows = (n_times + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
        if n_times == 1:
            axes = [axes]
        elif rows == 1:
            axes = axes.reshape(1, -1)
        
        for idx, ts in enumerate(selected_time_steps):
            row = idx // cols
            col = idx % cols
            ax = axes[row, col] if rows > 1 else axes[col]
            
            if ts in time_series_data and var_name in time_series_data[ts]['variables']:
                data = time_series_data[ts]
                var_data = data['variables'][var_name]
                
                if isinstance(var_data, dict):
                    # 2D variable structure
                    z_data = var_data['data']
                    grid_x = var_data['grid_x']
                    grid_y = var_data['grid_y']
                    
                    # Create 2D heatmap
                    if z_data is not None and z_data.ndim == 2:
                        # Try to reshape grid coordinates to match data
                        if len(grid_x) * len(grid_y) == z_data.size:
                            # Coordinates are 1D, create meshgrid
                            X, Y = np.meshgrid(grid_x, grid_y)
                            # Set explicit axis limits BEFORE plotting
                            ax.set_xlim(grid_x.min(), grid_x.max())
                            ax.set_ylim(grid_y.min(), grid_y.max())
                            im = ax.contourf(X, Y, z_data, levels=20, cmap='viridis')
                        elif grid_x.shape == z_data.shape and grid_y.shape == z_data.shape:
                            # Coordinates are 2D
                            ax.set_xlim(grid_x.min(), grid_x.max())
                            ax.set_ylim(grid_y.min(), grid_y.max())
                            im = ax.contourf(grid_x, grid_y, z_data, levels=20, cmap='viridis')
                        else:
                            # Fallback to simple imshow
                            ax.set_xlim(grid_x.min(), grid_x.max())
                            ax.set_ylim(grid_y.min(), grid_y.max())
                            im = ax.imshow(z_data, origin='lower', cmap='viridis', aspect='auto')
                        
                        ax.set_title(f't={ts}')
                        ax.set_xlabel('X')
                        ax.set_ylabel('Y')
                        
                        # Add colorbar
                        plt.colorbar(im, ax=ax, shrink=0.8)
                    else:
                        ax.text(0.5, 0.5, f'Invalid data\nshape: {z_data.shape if z_data is not None else "None"}', 
                               ha='center', va='center', transform=ax.transAxes)
                        ax.set_title(f't={ts} (error)')
                else:
                    ax.text(0.5, 0.5, 'No 2D data', ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(f't={ts} (no data)')
            else:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f't={ts} (missing)')
        
        # Hide unused subplots
        for idx in range(n_times, rows * cols):
            row = idx // cols
            col = idx % cols
            if rows > 1:
                axes[row, col].set_visible(False)
            else:
                axes[col].set_visible(False)
        
        plt.suptitle(f'2D Evolution of {var_name}', fontsize=14, y=0.98)
        plt.tight_layout()
        plt.show()
    
    print(f"✅ 2D time series plots created for {n_vars} variables")

def create_interactive_time_slider(time_series_data):
    """Create an interactive slider to view individual time steps (handles 1D and 2D)"""
    try:
        from ipywidgets import interact, SelectionSlider
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("❌ ipywidgets not available. Install with: pip install ipywidgets")
        return
    
    if not time_series_data:
        print("No time series data available for interactive viewing")
        return
    
    # Get available time steps and variables
    time_steps = sorted(time_series_data.keys())
    all_variables = set()
    for ts_data in time_series_data.values():
        all_variables.update(ts_data['variables'].keys())
    all_variables = sorted(list(all_variables))
    
    # Check if this is 2D data
    sample_data = next(iter(time_series_data.values()))
    is_2d = sample_data.get('dimensionality') == '2D'
    
    print(f"🎛️  Interactive {'2D' if is_2d else '1D'} time step viewer ready!")
    print(f"   Time steps: {len(time_steps)} ({min(time_steps)} to {max(time_steps)})")
    print(f"   Variables: {len(all_variables)} ({', '.join(all_variables)})")
    
    def plot_timestep(time_step):
        """Plot data for a specific time step (1D or 2D)"""
        if time_step not in time_series_data:
            print(f"No data available for time step {time_step}")
            return
        
        data = time_series_data[time_step]
        variables = data['variables']
        
        if not variables:
            print(f"No variables found for time step {time_step}")
            return
        
        if is_2d:
            plot_2d_timestep(time_step, data, variables)
        else:
            plot_1d_timestep(time_step, data, variables)
    
    def plot_1d_timestep(time_step, data, variables):
        """Plot 1D data for a specific time step"""
        grid = data['grid']
        
        # Create subplots for all variables
        n_vars = len(variables)
        fig, axes = plt.subplots(n_vars, 1, figsize=(12, 3*n_vars))
        if n_vars == 1:
            axes = [axes]
        
        for i, (var_name, var_data) in enumerate(sorted(variables.items())):
            ax = axes[i]
            
            # Use grid data if available and same length
            if grid is not None and len(grid) == len(var_data):
                x_data = grid
                ax.set_xlabel('Grid Position')
            else:
                x_data = range(len(var_data))
                ax.set_xlabel('Grid Index')
            
            # Plot the variable
            ax.plot(x_data, var_data, 'b-', linewidth=2)
            ax.set_ylabel(var_name)
            ax.grid(True, alpha=0.3)
            
            # Add range info
            if hasattr(var_data, 'min'):
                var_min, var_max = float(var_data.min()), float(var_data.max())
                ax.text(0.02, 0.98, f"Range: [{var_min:.3e}, {var_max:.3e}]", 
                       transform=ax.transAxes, va='top', fontsize=10,
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.suptitle(f'Time Step: {time_step}', fontsize=14, y=0.98)
        plt.tight_layout()
        plt.show()
    
    def plot_2d_timestep(time_step, data, variables):
        """Plot 2D data for a specific time step"""
        # Create subplots for all variables
        n_vars = len(variables)
        cols = min(2, n_vars)
        rows = (n_vars + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
        if n_vars == 1:
            axes = [axes]
        elif rows == 1:
            axes = axes.reshape(1, -1)
        
        for i, (var_name, var_data) in enumerate(sorted(variables.items())):
            row = i // cols
            col = i % cols
            ax = axes[row, col] if rows > 1 else axes[col]
            
            if isinstance(var_data, dict):
                # 2D variable structure
                z_data = var_data['data']
                grid_x = var_data['grid_x']
                grid_y = var_data['grid_y']
                
                # Create 2D heatmap
                if z_data is not None and z_data.ndim == 2:
                    # Try to reshape grid coordinates to match data
                    if len(grid_x) * len(grid_y) == z_data.size:
                        # Coordinates are 1D, create meshgrid
                        X, Y = np.meshgrid(grid_x, grid_y)
                        # Set explicit axis limits BEFORE plotting
                        ax.set_xlim(grid_x.min(), grid_x.max())
                        ax.set_ylim(grid_y.min(), grid_y.max())
                        im = ax.contourf(X, Y, z_data, levels=20, cmap='viridis')
                    elif grid_x.shape == z_data.shape and grid_y.shape == z_data.shape:
                        # Coordinates are 2D
                        ax.set_xlim(grid_x.min(), grid_x.max())
                        ax.set_ylim(grid_y.min(), grid_y.max())
                        im = ax.contourf(grid_x, grid_y, z_data, levels=20, cmap='viridis')
                    else:
                        # Fallback to simple imshow
                        ax.set_xlim(grid_x.min(), grid_x.max())
                        ax.set_ylim(grid_y.min(), grid_y.max())
                        im = ax.imshow(z_data, origin='lower', cmap='viridis', aspect='auto')
                    
                    ax.set_title(var_name)
                    ax.set_xlabel('X')
                    ax.set_ylabel('Y')
                    
                    # Add colorbar
                    plt.colorbar(im, ax=ax, shrink=0.8)
                    
                    # Add range info
                    if hasattr(z_data, 'min'):
                        var_min, var_max = float(z_data.min()), float(z_data.max())
                        ax.text(0.02, 0.98, f"Range: [{var_min:.3e}, {var_max:.3e}]", 
                               transform=ax.transAxes, va='top', fontsize=10,
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                else:
                    ax.text(0.5, 0.5, f'Invalid data\nshape: {z_data.shape if z_data is not None else "None"}', 
                           ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(f'{var_name} (error)')
            else:
                ax.text(0.5, 0.5, 'No 2D data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'{var_name} (no data)')
        
        # Hide unused subplots
        for i in range(n_vars, rows * cols):
            row = i // cols
            col = i % cols
            if rows > 1:
                axes[row, col].set_visible(False)
            else:
                axes[col].set_visible(False)
        
        plt.suptitle(f'Time Step: {time_step}', fontsize=14, y=0.98)
        plt.tight_layout()
        plt.show()
    
    # Create interactive slider that only allows actual time steps
    time_slider = SelectionSlider(
        options=time_steps,
        value=time_steps[0],
        description='Time Step:',
        style={'description_width': 'initial'},
        layout={'width': '500px'}
    )
    
    # Use interact to connect slider to plotting function
    interact(plot_timestep, time_step=time_slider)

def analyze_results(run_result):
    """Analyze and visualize MFC results with Silo (if available) or basic file analysis"""
    if not run_result or not run_result['success']:
        print("No successful run to analyze")
        return
    
    print(f"=== Results Analysis ===")
    print(f"Runtime: {run_result['runtime']:.2f} seconds")
    
    example_dir = Path(run_result['example_dir'])
    
    # Find MFC Silo output files (they're in subdirectories)
    output_files = {'silo': [], 'collection': [], 'other': []}
    
    for output_dir in ['silo_hdf5', 'D', 'hdf5', 'silo']:
        dir_path = example_dir / output_dir
        if dir_path.exists():
            print(f"📁 Checking {output_dir}/ directory...")
            
            # Look for collection files in root/ subdirectory (these are the main files)
            root_dir = dir_path / 'root'
            if root_dir.exists():
                collection_files = list(root_dir.glob('collection_*.silo'))
                if collection_files:
                    output_files['collection'].extend(collection_files)
                    print(f"   Found {len(collection_files)} collection files in root/")
                    for f in collection_files[:3]:  # Show first 3
                        print(f"     • {f.name} ({f.stat().st_size/1024:.1f} KB)")
                    if len(collection_files) > 3:
                        print(f"     ... and {len(collection_files) - 3} more")
            
            # Also check for individual rank files and other silo files
            for item in dir_path.iterdir():
                if item.is_file() and item.suffix in ['.silo', '.h5', '.hdf5']:
                    output_files['silo'].append(item)
                elif item.is_dir():
                    # Check subdirectories for .silo files
                    silo_files = list(item.glob('*.silo'))
                    if silo_files:
                        output_files['silo'].extend(silo_files[:5])  # Limit to avoid spam
                        print(f"   Found {len(silo_files)} files in {item.name}/")
    
    total_silo = len(output_files['silo']) + len(output_files['collection'])
    print(f"Found {len(output_files['collection'])} collection files, {len(output_files['silo'])} other Silo files")
    
    if not output_files['collection'] and not output_files['silo']:
        print("❌ No Silo files found for analysis!")
        print("   Make sure post_process was included in the run stages.")
        print("   Expected structure: silo_hdf5/root/collection_*.silo")
        return
    
    # Use HDF5 analysis for MFC collection files (has better variable mapping)
    print("🔧 Using HDF5 analysis for MFC file analysis...")
    try:
        # Use collection files if available (these are the master files)
        if output_files['collection']:
            analyze_mfc_hdf5_silo(output_files['collection'][0])
        elif output_files['silo']:
            analyze_mfc_hdf5_silo(output_files['silo'][0])
        else:
            print("❌ No Silo files to analyze")
    except Exception as e:
        print(f"⚠️ HDF5 analysis failed: {e}")
        if PACKAGES.get('silo'):
            print("   Trying Silo library analysis as fallback...")
            try:
                if output_files['collection']:
                    analyze_with_silo(output_files['collection'][0], example_dir, mfc_python)
                elif output_files['silo']:
                    analyze_with_silo(output_files['silo'][0], example_dir, mfc_python)
            except Exception as e2:
                print(f"   Silo analysis also failed: {e2}")
                print("   Falling back to basic analysis...")
                all_files = output_files['collection'] + output_files['silo']
                analyze_basic(all_files, example_dir)
        else:
            print("   Silo not available, falling back to basic analysis...")
            all_files = output_files['collection'] + output_files['silo']
            analyze_basic(all_files, example_dir)

def analyze_with_silo(file_path, example_dir, python_exe):
    """Analyze MFC files using Silo library in MFC's Python environment"""
    print(f"📊 Analyzing with Silo: {file_path.name}")
    
    # Create a script to run Silo analysis in MFC's Python for MFC collection files
    analysis_script = f'''
import sys
sys.path.insert(0, "/")  # Ensure we can import Silo
import Silo
import numpy as np

try:
    # Open Silo collection file
    db = Silo.Open("{file_path}", Silo.DB_READ)
    
    # Get table of contents
    toc = db.GetToc()
    
    print(f"File type: Collection file (MFC parallel output)")
    print(f"Variables: {{len(toc.var_names) if toc.var_names else 0}}")
    print(f"Quad meshes: {{len(toc.qmesh_names) if toc.qmesh_names else 0}}")
    print(f"Multimeshes: {{len(toc.multimesh_names) if toc.multimesh_names else 0}}")
    print(f"Multivars: {{len(toc.multivar_names) if toc.multivar_names else 0}}")
    
    # For MFC collection files, we want to look at multivars (these combine data from all ranks)
    if toc.multivar_names:
        print("Available multivars (combined from all ranks):")
        for var_name in toc.multivar_names[:10]:
            print(f"  • {{var_name}}")
        if len(toc.multivar_names) > 10:
            print(f"  ... and {{len(toc.multivar_names) - 10}} more")
        
        # Try to analyze first multivar
        var_name = toc.multivar_names[0]
        try:
            print(f"Analyzing multivar: {{var_name}}")
            multivar_info = db.GetMultivar(var_name)
            print(f"  Type: {{type(multivar_info)}}")
            print(f"  Info: {{multivar_info}}")
            
            # For collection files, we might need to read individual blocks
            # This is more complex - just show the structure for now
            
        except Exception as e:
            print(f"Could not read multivar {{var_name}}: {{e}}")
    
    elif toc.var_names:
        print("Available variables:")
        for var_name in toc.var_names[:10]:
            print(f"  • {{var_name}}")
        if len(toc.var_names) > 10:
            print(f"  ... and {{len(toc.var_names) - 10}} more")
        
        # Try to get basic info about first variable
        var_name = toc.var_names[0]
        try:
            var_data = db.GetVar(var_name)
            data = np.array(var_data)
            print(f"Sample variable {{var_name}}: shape {{data.shape}}, dtype {{data.dtype}}")
            if data.size > 0:
                print(f"  Min: {{data.min():.3e}}, Max: {{data.max():.3e}}")
        except Exception as e:
            print(f"Could not analyze {{var_name}}: {{e}}")
    
    else:
        print("No direct variables found - this is typical for collection files")
        print("Collection files reference data in individual rank files")
    
    db.Close()
    print("✅ Silo collection analysis completed successfully")
    
except Exception as e:
    print(f"❌ Silo analysis error: {{e}}")
    import traceback
    traceback.print_exc()
    sys.exit(1)
'''
    
    # Run the analysis script in MFC's Python environment
    try:
        result = subprocess.run([python_exe, "-c", analysis_script],
                              capture_output=True, text=True, timeout=30)
        
        if result.returncode == 0:
            print(result.stdout)
        else:
            print(f"Silo analysis failed: {result.stderr}")
            raise Exception("Silo script failed")
            
    except Exception as e:
        raise Exception(f"Failed to run Silo analysis: {e}")

def analyze_mfc_hdf5_silo(file_path):
    """Analyze MFC's HDF5-backed Silo files using h5py when Silo library fails"""
    if not PACKAGES.get('h5py'):
        raise Exception("h5py not available for HDF5 analysis")
    
    import h5py
    print(f"📊 Analyzing MFC HDF5-Silo file: {file_path.name}")
    
    try:
        with h5py.File(file_path, 'r') as f:
            print(f"   File size: {file_path.stat().st_size/1024:.1f} KB")
            print(f"   HDF5 groups: {len(list(f.keys()))}")
            
            # First, find variable names and their dataset mappings
            variable_mappings = {}
            
            # Extract MFC variable names and their dataset mappings
            for key in f.keys():
                try:
                    group = f[key]
                    if hasattr(group, 'attrs') and 'silo' in group.attrs:
                        # Extract the silo attribute which contains grid and data references
                        silo_attr = group.attrs['silo']
                        
                        # Handle numpy.void (structured array) format
                        if hasattr(silo_attr, 'item') and len(silo_attr.item()) >= 3:
                            # Convert numpy.void to tuple: (npts, grid_ref, data_ref)
                            silo_tuple = silo_attr.item()
                            grid_ref = silo_tuple[1].decode('utf-8') if isinstance(silo_tuple[1], bytes) else str(silo_tuple[1])
                            data_ref = silo_tuple[2].decode('utf-8') if isinstance(silo_tuple[2], bytes) else str(silo_tuple[2])
                            variable_mappings[data_ref] = key  # Map dataset path to variable name
                            print(f"   Variable '{key}': grid={grid_ref}, data={data_ref}")
                        elif isinstance(silo_attr, (list, tuple)) and len(silo_attr) >= 3:
                            # Fallback to list/tuple format: [npts, grid_ref, data_ref]
                            grid_ref = silo_attr[1].decode('utf-8') if isinstance(silo_attr[1], bytes) else str(silo_attr[1])
                            data_ref = silo_attr[2].decode('utf-8') if isinstance(silo_attr[2], bytes) else str(silo_attr[2])
                            variable_mappings[data_ref] = key  # Map dataset path to variable name
                            print(f"   Variable '{key}': grid={grid_ref}, data={data_ref}")
                except:
                    continue
            
            # Now collect datasets with enhanced information
            found_vars = []
            var_info = []
            
            def find_datasets(name, obj):
                if isinstance(obj, h5py.Dataset):
                    found_vars.append(name)
                    shape = obj.shape
                    dtype = obj.dtype
                    
                    # Look up the variable name for this dataset
                    # Try both with and without leading slash to match mapping
                    var_name = variable_mappings.get(name, variable_mappings.get('/' + name, name))
                    
                    if obj.size > 0:
                        try:
                            data_sample = obj[...]
                            if hasattr(data_sample, 'min'):
                                var_info.append({
                                    'name': name,
                                    'var_name': var_name,  # Add the MFC variable name
                                    'shape': shape,
                                    'dtype': str(dtype),
                                    'min': float(data_sample.min()),
                                    'max': float(data_sample.max()),
                                    'data': data_sample
                                })
                        except:
                            var_info.append({
                                'name': name,
                                'var_name': var_name,
                                'shape': shape,
                                'dtype': str(dtype),
                                'min': None,
                                'max': None,
                                'data': None
                            })
            
            f.visititems(find_datasets)
            
            print(f"   Found {len(found_vars)} datasets:")
            for var in var_info[:10]:  # Show first 10
                if var['min'] is not None:
                    print(f"     • {var['name']}: {var['shape']} {var['dtype']} [{var['min']:.3e}, {var['max']:.3e}]")
                else:
                    print(f"     • {var['name']}: {var['shape']} {var['dtype']}")
            
            if len(found_vars) > 10:
                print(f"     ... and {len(found_vars) - 10} more")
            
            # Try to plot some data if matplotlib is available
            if PACKAGES.get('matplotlib') and var_info:
                plot_mfc_data(var_info, file_path.name)
            
            print("✅ HDF5-Silo analysis completed successfully")
            
    except Exception as e:
        raise Exception(f"Failed to read HDF5-Silo file: {e}")

def plot_mfc_data(var_info, filename):
    """Plot MFC data from HDF5-Silo analysis with proper grid as x-axis"""
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Find the grid data by looking for monotonically increasing coordinates
    grid_data = None
    variables = []
    
    # First, collect all suitable 1D datasets
    candidates = []
    for var in var_info:
        if var['data'] is not None and len(var['shape']) == 1 and var['shape'][0] < 10000:
            candidates.append(var)
    
    # Use the variable mappings we extracted to find the correct grid and variables
    silo_datasets = [var for var in candidates if '.silo/#' in var['name']]
    
    grid_data = None
    variables = []
    
    if silo_datasets:
        # Find datasets that have proper variable names (mapped from HDF5 attributes)
        mapped_variables = []
        unmapped_datasets = []
        
        for dataset in silo_datasets:
            if hasattr(dataset, 'var_name') and dataset['var_name'] != dataset['name']:
                # This dataset has a proper MFC variable name
                mapped_variables.append(dataset)
                print(f"   Mapped variable: {dataset['var_name']} -> {dataset['name']}")
            else:
                unmapped_datasets.append(dataset)
        
        if mapped_variables:
            # Use the first unmapped dataset as grid (these are usually the grid coordinates)
            if unmapped_datasets:
                grid_data = unmapped_datasets[0]
            else:
                # Fallback: use first dataset as grid
                grid_data = silo_datasets[0]
            
            variables = mapped_variables
        else:
            # No mapped variables found, fall back to old pairing logic
            grid_data = silo_datasets[0]
            variables = silo_datasets[1::2]  # Every other dataset starting from index 1
    
    # Fallback to old method if no clear Silo pattern
    if not grid_data:
        # Fallback to naming convention
        for var in candidates:
            if '#000001' in var['name'] or 'grid' in var['name'].lower() or 'coord' in var['name'].lower():
                grid_data = var
                variables = [v for v in candidates if v != grid_data]
                break
        
        # Final fallback: use first dataset as grid
        if grid_data is None and candidates:
            grid_data = candidates[0]
            variables = candidates[1:]
    
    if not grid_data:
        print("   No suitable grid data found for x-axis")
        return
        
    if not variables:
        print("   No suitable variable data found for plotting")
        return
    
    # Plot all variables against the grid
    n_plots = len(variables)
    if n_plots > 0:
        fig, axes = plt.subplots(n_plots, 1, figsize=(12, 4*n_plots))
        if n_plots == 1:
            axes = [axes]
        
        for i, var in enumerate(variables):
            # Use grid data as x-axis and variable data as y-axis
            x_data = grid_data['data']
            y_data = var['data']
            
            # Make sure arrays are the same length
            if len(x_data) == len(y_data):
                axes[i].plot(x_data, y_data, 'b-', linewidth=2)
                
                # Use MFC variable name if available, otherwise use dataset name
                var_label = var.get('var_name', var['name'])
                dataset_name = var['name'].split('/')[-1] if '/' in var['name'] else var['name']
                
                axes[i].set_ylabel(var_label)
                axes[i].set_xlabel('Grid Position')
                axes[i].grid(True, alpha=0.3)
                
                if var['min'] is not None:
                    axes[i].text(0.02, 0.98, f"Range: [{var['min']:.3e}, {var['max']:.3e}]", 
                               transform=axes[i].transAxes, va='top', fontsize=10,
                               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
            else:
                print(f"   Warning: Grid length ({len(x_data)}) != Variable length ({len(y_data)}) for {var['name']}")
                # Fall back to index plotting
                axes[i].plot(range(len(y_data)), y_data, 'b-', linewidth=2)
                
                var_label = var.get('var_name', var['name'])
                axes[i].set_ylabel(var_label)
                axes[i].set_xlabel('Index')
                axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"   ✅ Plotted {n_plots} variables")

def analyze_basic(silo_files, example_dir):
    """Basic file analysis when Silo is not available"""
    print(f"📊 Basic file analysis of {len(silo_files)} files:")
    
    total_size = 0
    for file in silo_files:
        size = file.stat().st_size
        total_size += size
        print(f"   {file.name}: {size/1024:.1f} KB")
    
    print(f"Total output size: {total_size/1024:.1f} KB")
    
    # Try to plot with h5py if available and files are HDF5
    if PACKAGES.get('matplotlib') and PACKAGES.get('h5py'):
        hdf5_files = [f for f in silo_files if f.suffix in ['.h5', '.hdf5']]
        if hdf5_files:
            try:
                plot_basic_hdf5(hdf5_files[0])
            except Exception as e:
                print(f"⚠️ Could not plot HDF5 data: {e}")

def plot_basic_hdf5(file_path):
    """Basic HDF5 plotting if the file is actually HDF5 format"""
    import matplotlib.pyplot as plt
    import h5py
    
    print(f"🎯 Attempting basic HDF5 plot: {file_path.name}")
    
    try:
        with h5py.File(file_path, 'r') as f:
            # List datasets
            datasets = []
            def find_datasets(name, obj):
                if isinstance(obj, h5py.Dataset):
                    datasets.append(name)
            f.visititems(find_datasets)
            
            if datasets:
                print(f"   Found {len(datasets)} datasets")
                # Try to plot first dataset
                data = f[datasets[0]][:]
                if len(data.shape) == 1 and len(data) > 1:
                    plt.figure(figsize=(10, 6))
                    plt.plot(data)
                    plt.title(f'{file_path.name}: {datasets[0]}')
                    plt.xlabel('Index')
                    plt.ylabel('Value')
                    plt.grid(True, alpha=0.3)
                    plt.show()
                    print("✅ Basic plot generated")
                else:
                    print(f"   Data shape {data.shape} - not suitable for basic line plot")
            else:
                print("   No plottable datasets found")
                
    except Exception as e:
        print(f"   Not a valid HDF5 file or read error: {e}")

In [ ]:
## 4. Run a Demo Example
# Let's run a simple 1D example to test the complete pipeline:

# Run a demo example if build was successful
try:
    if build_success:
        selected_example = "2D_advection"
        print(f"  🚀 Running demo example: {selected_example}")
        print("   (Including post-processing for Silo file generation)")
        
        # Run the example with all stages
        run_result = run_example(selected_example, num_procs=2, timeout=300000)
        
        if run_result and run_result['success']:
            print(f"\n✅ Demo completed successfully!")
            print("   Ready for analysis in next cell")
        else:
            print(f"⚠️  Demo failed or timed out")
            run_result = None
    else:
        print("❌ Cannot run examples - MFC build failed")
        print("   Make sure the build cell ran successfully first")
        run_result = None
        
except NameError:
    print("❌ Build variables not found - run previous cells first")
    run_result = None

# Make run_result available for analysis
if 'run_result' not in locals():
    run_result = None


In [ ]:
## 5. Time Series Analysis
## Analyze how variables evolve across multiple time steps

try:
    if run_result and run_result['success']:
        print("🔍 Analyzing time series evolution...")
        
        # Option 1: Analyze all available time steps (automatically selects subset)
        time_data = analyze_time_series(run_result)
        
        # Option 2: Analyze specific time steps (uncomment to use)
        # specific_steps = [0, 200, 400, 600]  # Specify which time steps you want
        # time_data = analyze_time_series(run_result, selected_timesteps=specific_steps)
        
        if time_data:
            print(f"\n📈 Time series analysis complete!")
            print(f"   Time steps analyzed: {sorted(time_data.keys())}")
            print(f"   Variables tracked: {sorted(list(time_data[list(time_data.keys())[0]]['variables'].keys()))}")
            print(f"   💡 Tip: Modify 'selected_timesteps' parameter to choose specific time steps")
        else:
            print("❌ No time series data extracted")
            
    elif run_result:
        print("❌ Cannot analyze time series - demo run failed")
        print(f"   Error: {run_result.get('stderr', 'Unknown error')}")
    else:
        print("❌ No results to analyze")
        print("   Make sure the previous cell ran an example successfully")
       
except NameError:
    print("❌ No run_result found - run the demo cell first")
    
except Exception as e:
    print(f"❌ Time series analysis failed: {e}")
    print("   Check if the demo run generated multiple time step files")


In [ ]:
## 6. Single Timestep Analysis
## Analyze the latest timestep from the demo run using our enhanced 2D analysis:

def analyze_latest_timestep(run_result):
    """Analyze the most recent timestep with enhanced 2D support"""
    if not run_result or not run_result['success']:
        print("No successful run to analyze")
        return
    
    print(f"=== Latest Timestep Analysis ===")
    print(f"Runtime: {run_result['runtime']:.2f} seconds")
    
    example_dir = Path(run_result['example_dir'])
    
    # Find collection files
    collection_files = []
    for output_dir in ['silo_hdf5', 'silo', 'hdf5']:
        dir_path = example_dir / output_dir
        if dir_path.exists():
            root_dir = dir_path / 'root'
            if root_dir.exists():
                files = list(root_dir.glob('collection_*.silo'))
                collection_files.extend(files)
    
    if not collection_files:
        print("❌ No collection files found for analysis")
        return
    
    # Get the latest timestep file
    time_steps = []
    file_map = {}
    for file in collection_files:
        try:
            time_str = file.stem.split('_')[1]
            time_step = int(time_str)
            time_steps.append(time_step)
            file_map[time_step] = file
        except:
            continue
    
    if not time_steps:
        print("❌ No valid timestep files found")
        return
    
    latest_timestep = max(time_steps)
    latest_file = file_map[latest_timestep]
    
    print(f"📊 Analyzing latest timestep: {latest_timestep}")
    print(f"   File: {latest_file.name}")
    
    # Use our enhanced analysis
    try:
        data = analyze_single_timestep(latest_file)
        if data:
            dimensionality = data.get('dimensionality', '1D')
            num_vars = len(data['variables'])
            
            print(f"✅ Analysis complete!")
            print(f"   📐 Dimensionality: {dimensionality}")
            print(f"   🔢 Variables found: {num_vars}")
            print(f"   📋 Variable list: {sorted(data['variables'].keys())}")
            
            # Create a quick visualization of the latest timestep
            if data['variables'] and PACKAGES.get('matplotlib'):
                print(f"\n📊 Creating visualization...")
                
                if dimensionality == '2D':
                    create_2d_snapshot(data, latest_timestep)
                else:
                    create_1d_snapshot(data, latest_timestep)
            
            return data
        else:
            print("❌ Failed to extract data from latest timestep")
            return None
            
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        return None

def create_2d_snapshot(data, timestep):
    """Create a 2D snapshot visualization"""
    import matplotlib.pyplot as plt
    import numpy as np
    
    variables = data['variables']
    n_vars = len(variables)
    
    if n_vars == 0:
        print("No variables to plot")
        return
    
    # Show first 4 variables if there are many
    vars_to_plot = list(variables.keys())[:4]
    n_plots = len(vars_to_plot)
    
    cols = min(2, n_plots)
    rows = (n_plots + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
    if n_plots == 1:
        axes = [axes]
    elif rows == 1:
        axes = axes.reshape(1, -1)
    
    for i, var_name in enumerate(vars_to_plot):
        row = i // cols
        col = i % cols
        ax = axes[row, col] if rows > 1 else axes[col]
        
        var_data = variables[var_name]
        
        if isinstance(var_data, dict):
            z_data = var_data['data']
            grid_x = var_data['grid_x']
            grid_y = var_data['grid_y']
            
            if z_data is not None and z_data.ndim == 2:
                try:
                    if len(grid_x) * len(grid_y) == z_data.size:
                        # Coordinates are 1D, create meshgrid
                        X, Y = np.meshgrid(grid_x, grid_y)
                        # Set explicit axis limits BEFORE plotting
                        ax.set_xlim(grid_x.min(), grid_x.max())
                        ax.set_ylim(grid_y.min(), grid_y.max())
                        im = ax.contourf(X, Y, z_data, levels=20, cmap='viridis')
                    elif grid_x.shape == z_data.shape and grid_y.shape == z_data.shape:
                        # Coordinates are 2D
                        ax.set_xlim(grid_x.min(), grid_x.max())
                        ax.set_ylim(grid_y.min(), grid_y.max())
                        im = ax.contourf(grid_x, grid_y, z_data, levels=20, cmap='viridis')
                    else:
                        # Fallback to simple imshow
                        ax.set_xlim(grid_x.min(), grid_x.max())
                        ax.set_ylim(grid_y.min(), grid_y.max())
                        im = ax.imshow(z_data, origin='lower', cmap='viridis', aspect='auto')
                    
                    ax.set_title(f'{var_name}')
                    ax.set_xlabel('X')
                    ax.set_ylabel('Y')
                    plt.colorbar(im, ax=ax, shrink=0.8)
                    
                except Exception as e:
                    ax.text(0.5, 0.5, f'Plot error:\n{str(e)[:50]}...', 
                           ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(f'{var_name} (error)')
            else:
                ax.text(0.5, 0.5, 'Invalid 2D data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'{var_name} (invalid)')
        else:
            ax.text(0.5, 0.5, 'Not 2D format', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{var_name} (format error)')
    
    # Hide unused subplots
    for i in range(n_plots, rows * cols):
        row = i // cols
        col = i % cols
        if rows > 1:
            axes[row, col].set_visible(False)
        else:
            axes[col].set_visible(False)
    
    plt.suptitle(f'2D Snapshot - Timestep {timestep}', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()
    
    print(f"✅ 2D snapshot created for timestep {timestep}")

def create_1d_snapshot(data, timestep):
    """Create a 1D snapshot visualization"""
    import matplotlib.pyplot as plt
    
    variables = data['variables']
    grid = data['grid']
    
    if not variables:
        print("No variables to plot")
        return
    
    n_vars = len(variables)
    fig, axes = plt.subplots(n_vars, 1, figsize=(12, 3*n_vars))
    if n_vars == 1:
        axes = [axes]
    
    for i, (var_name, var_data) in enumerate(sorted(variables.items())):
        ax = axes[i]
        
        if grid is not None and len(grid) == len(var_data):
            x_data = grid
            ax.set_xlabel('Grid Position')
        else:
            x_data = range(len(var_data))
            ax.set_xlabel('Grid Index')
        
        ax.plot(x_data, var_data, 'b-', linewidth=2)
        ax.set_ylabel(var_name)
        ax.grid(True, alpha=0.3)
        
        if hasattr(var_data, 'min'):
            var_min, var_max = float(var_data.min()), float(var_data.max())
            ax.text(0.02, 0.98, f"Range: [{var_min:.3e}, {var_max:.3e}]", 
                   transform=ax.transAxes, va='top', fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.suptitle(f'1D Snapshot - Timestep {timestep}', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()
    
    print(f"✅ 1D snapshot created for timestep {timestep}")

try:
    if run_result and run_result['success']:
        print("🔍 Analyzing latest timestep...")
        snapshot_data = analyze_latest_timestep(run_result)
        
        if snapshot_data:
            print(f"\n💡 For complete time evolution analysis, use Cell 5 (Time Series Analysis)")
            print(f"💡 For interactive exploration, use Cell 7 (Interactive Analysis)")
        
    elif run_result:
        print("❌ Cannot analyze - demo run failed")
        print(f"   Error: {run_result.get('stderr', 'Unknown error')}")
    else:
        print("❌ No results to analyze")
        print("   Make sure the previous cell ran an example successfully")
        
except NameError:
    print("❌ No run_result found - run the demo cell first")
    
except Exception as e:
    print(f"❌ Analysis failed: {e}")
    print("   Check if the demo run generated output files")

In [ ]:
## 7. Interactive Time Series Analysis
## Analyze how variables evolve across multiple time steps with an interactive slider

try:
    if run_result and run_result['success']:
        print("🔍 Analyzing time series evolution...")
        
        # Option 1: Analyze all available time steps (automatically selects subset)
        time_data = analyze_time_series(run_result)
        
        # Option 2: Analyze specific time steps (uncomment to use)
        # specific_steps = [0, 200, 400, 600]  # Specify which time steps you want
        # time_data = analyze_time_series(run_result, selected_timesteps=specific_steps)
        
        if time_data:
            print(f"\n📈 Time series analysis complete!")
            print(f"   Time steps analyzed: {sorted(time_data.keys())}")
            print(f"   Variables tracked: {sorted(list(time_data[list(time_data.keys())[0]]['variables'].keys()))}")
            
            # Create interactive slider for individual time step viewing
            print(f"\n🎛️  Creating interactive time step viewer...")
            create_interactive_time_slider(time_data)
            
        else:
            print("❌ No time series data extracted")
            
    elif run_result:
        print("❌ Cannot analyze time series - demo run failed")
        print(f"   Error: {run_result.get('stderr', 'Unknown error')}")
    else:
        print("❌ No results to analyze")
        print("   Make sure the previous cell ran an example successfully")
        
except NameError:
    print("❌ No run_result found - run the demo cell first")
    
except Exception as e:
    print(f"❌ Time series analysis failed: {e}")
    print("   Check if the demo run generated multiple time step files")
